# Initial Multimodal screen: results analysis

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json

import pandas as pd
from IPython.display import display

from src.models.classical_baseline import HORIZONS

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'multimodal'
RESULTS_DIR = EXPERIMENT_DIR / 'results'

MODEL_FILES = {
    'CGM-only':         RESULTS_DIR / 'cgm_only' / 'cgm_only_val_stability_summary.json',
    'Gated_time':       RESULTS_DIR / 'gated_cgm_time' / 'gated_cgm_time_val_stability_summary.json',
    'Gated_resnet18':       RESULTS_DIR / 'gated_resnet18' / 'gated_cgm_time_resnet18_val_stability_summary.json',
    'Gated_vit':       RESULTS_DIR / 'gated_vit' / 'gated_cgm_time_vit_val_stability_summary.json',
    'Gated_clip':       RESULTS_DIR / 'gated_clip' / 'gated_cgm_time_clip_val_stability_summary.json',
    'Gated_dinov2':       RESULTS_DIR / 'gated_dinov2' / 'gated_cgm_time_dinov2_val_stability_summary.json',
}

METRICS = {
    'rmse': 'RMSE (mg/dL)', 'mae': 'MAE (mg/dL)', 'grmse': 'gRMSE',
    'zone_a': 'Zone A (%)', 'zone_ab': 'Zone A+B (%)',
}

In [3]:
per_horizon_summary = {}
for model_name, path in MODEL_FILES.items():
    if not path.exists():
        print(f'MISSING: {model_name} -- expected {path}')
        continue
    with open(path) as f:
        d = json.load(f)
    per_horizon_summary[model_name] = d['per_horizon_summary']
    seeds = d.get('seeds')
    print(f"{model_name:<16} seeds={seeds if seeds is not None else '(deterministic)'}")

CGM-only         seeds=[7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
Gated_time       seeds=[7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
Gated_resnet18   seeds=[7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
Gated_vit        seeds=[7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
Gated_clip       seeds=[7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
Gated_dinov2     seeds=[7, 14, 21, 28, 35, 42, 49, 56, 63, 70]


In [4]:
def format_cell(entry: dict) -> str:
    return (f"{entry['mean']:.2f} \u00b1 {entry['sd1_across_seeds']:.2f} "
            f"({entry['sd2_across_participants']:.2f})")

tables = {}
for metric_key, metric_label in METRICS.items():
    rows = {}
    for model_name in MODEL_FILES:
        if model_name not in per_horizon_summary:
            continue
        rows[model_name] = {
            f'{h}min': format_cell(per_horizon_summary[model_name][str(h)][metric_key])
            for h in HORIZONS
        }
    table = pd.DataFrame(rows).T
    table = table[[f'{h}min' for h in HORIZONS]]  # enforce horizon column order
    tables[metric_key] = table

    print(f'=== {metric_label}: mean \u00b1 sd1 (sd2) ===')
    display(table)
    print()

=== RMSE (mg/dL): mean ± sd1 (sd2) ===


,15min,30min,60min,90min,120min
CGM-only,10.70 ± 0.06 (2.10),17.01 ± 0.06 (3.21),24.40 ± 0.04 (6.82),28.88 ± 0.03 (10.16),31.57 ± 0.04 (12.08)
Gated_time,10.58 ± 0.07 (2.09),16.68 ± 0.07 (3.61),24.07 ± 0.06 (7.04),28.40 ± 0.05 (10.27),31.25 ± 0.06 (12.14)
Gated_resnet18,11.72 ± 0.51 (2.31),17.91 ± 0.20 (3.87),25.82 ± 0.18 (7.54),30.62 ± 0.15 (10.61),33.41 ± 0.14 (12.13)
Gated_vit,13.01 ± 0.31 (2.79),18.85 ± 0.16 (4.30),26.55 ± 0.16 (7.99),31.40 ± 0.20 (11.13),34.10 ± 0.21 (12.61)
Gated_clip,12.23 ± 0.50 (2.49),18.10 ± 0.17 (4.02),25.75 ± 0.18 (7.84),30.56 ± 0.14 (11.03),33.33 ± 0.09 (12.52)
Gated_dinov2,12.26 ± 0.69 (2.58),18.38 ± 0.25 (4.35),26.44 ± 0.26 (7.99),31.24 ± 0.20 (10.97),33.88 ± 0.30 (12.42)



=== MAE (mg/dL): mean ± sd1 (sd2) ===


,15min,30min,60min,90min,120min
CGM-only,7.26 ± 0.08 (1.42),11.46 ± 0.10 (2.12),16.34 ± 0.12 (4.31),19.49 ± 0.14 (6.59),21.62 ± 0.16 (8.06)
Gated_time,7.22 ± 0.09 (1.41),11.25 ± 0.08 (2.35),15.93 ± 0.11 (4.50),18.97 ± 0.11 (6.71),21.24 ± 0.12 (8.07)
Gated_resnet18,8.02 ± 0.35 (1.56),12.03 ± 0.13 (2.53),17.02 ± 0.18 (5.00),20.34 ± 0.20 (6.95),22.68 ± 0.31 (7.94)
Gated_vit,8.88 ± 0.45 (1.89),12.58 ± 0.28 (2.79),17.50 ± 0.34 (5.24),20.89 ± 0.44 (7.18),23.17 ± 0.51 (8.03)
Gated_clip,8.34 ± 0.36 (1.72),12.13 ± 0.14 (2.62),16.97 ± 0.16 (5.09),20.26 ± 0.22 (7.09),22.56 ± 0.11 (8.06)
Gated_dinov2,8.35 ± 0.47 (1.75),12.25 ± 0.18 (2.83),17.36 ± 0.14 (5.31),20.71 ± 0.15 (7.23),22.94 ± 0.22 (8.18)



=== gRMSE: mean ± sd1 (sd2) ===


,15min,30min,60min,90min,120min
CGM-only,11.57 ± 0.13 (2.86),19.02 ± 0.12 (4.83),28.73 ± 0.15 (10.24),34.80 ± 0.15 (14.90),38.34 ± 0.17 (17.74)
Gated_time,11.52 ± 0.14 (2.92),18.69 ± 0.15 (5.26),28.23 ± 0.16 (10.40),34.17 ± 0.17 (15.06),37.94 ± 0.18 (17.83)
Gated_resnet18,12.76 ± 0.48 (3.21),20.22 ± 0.24 (5.84),30.60 ± 0.34 (11.63),37.10 ± 0.33 (16.13),40.81 ± 0.34 (18.51)
Gated_vit,14.34 ± 0.26 (3.85),21.46 ± 0.25 (6.49),31.60 ± 0.19 (12.29),38.27 ± 0.27 (16.96),41.91 ± 0.25 (19.33)
Gated_clip,13.38 ± 0.55 (3.43),20.49 ± 0.27 (6.04),30.62 ± 0.25 (11.98),37.27 ± 0.16 (16.68),40.98 ± 0.20 (19.06)
Gated_dinov2,13.56 ± 0.80 (3.65),21.04 ± 0.34 (6.58),31.54 ± 0.29 (12.37),38.06 ± 0.22 (16.84),41.58 ± 0.38 (19.10)



=== Zone A (%): mean ± sd1 (sd2) ===


,15min,30min,60min,90min,120min
CGM-only,96.84 ± 0.10 (3.08),90.09 ± 0.13 (5.50),81.99 ± 0.19 (5.08),77.06 ± 0.35 (7.15),73.34 ± 0.33 (8.25)
Gated_time,97.05 ± 0.15 (2.73),90.36 ± 0.14 (4.94),82.34 ± 0.26 (4.45),77.80 ± 0.17 (7.00),73.76 ± 0.30 (8.22)
Gated_resnet18,95.85 ± 0.65 (3.91),89.18 ± 0.46 (5.37),80.44 ± 0.60 (5.80),74.96 ± 0.57 (8.38),70.80 ± 0.77 (8.86)
Gated_vit,94.31 ± 0.57 (5.43),88.14 ± 0.66 (6.06),79.76 ± 0.71 (6.79),74.47 ± 0.90 (8.63),70.39 ± 1.31 (9.09)
Gated_clip,95.14 ± 0.63 (4.65),88.92 ± 0.38 (5.70),80.65 ± 0.48 (6.22),75.26 ± 0.58 (8.32),71.30 ± 0.39 (9.07)
Gated_dinov2,95.31 ± 0.95 (4.49),88.73 ± 0.31 (5.76),79.88 ± 0.42 (6.53),74.53 ± 0.48 (8.75),70.62 ± 0.62 (9.31)



=== Zone A+B (%): mean ± sd1 (sd2) ===


,15min,30min,60min,90min,120min
CGM-only,99.65 ± 0.03 (0.67),99.37 ± 0.02 (0.97),98.77 ± 0.01 (1.30),98.27 ± 0.03 (1.82),97.94 ± 0.03 (2.31)
Gated_time,99.66 ± 0.01 (0.65),99.37 ± 0.02 (1.00),98.81 ± 0.02 (1.27),98.41 ± 0.02 (1.87),98.03 ± 0.03 (2.40)
Gated_resnet18,99.61 ± 0.07 (0.73),99.30 ± 0.02 (1.06),98.70 ± 0.03 (1.40),98.14 ± 0.04 (2.00),97.79 ± 0.06 (2.48)
Gated_vit,99.48 ± 0.02 (0.95),99.19 ± 0.02 (1.15),98.59 ± 0.02 (1.52),98.06 ± 0.02 (2.13),97.67 ± 0.03 (2.73)
Gated_clip,99.51 ± 0.06 (0.91),99.26 ± 0.01 (1.10),98.69 ± 0.02 (1.40),98.11 ± 0.03 (2.06),97.73 ± 0.03 (2.61)
Gated_dinov2,99.56 ± 0.10 (0.81),99.22 ± 0.01 (1.11),98.59 ± 0.09 (1.48),98.07 ± 0.06 (2.06),97.76 ± 0.05 (2.53)


In [5]:
rmse_means = pd.DataFrame({
    model_name: {f'{h}min': per_horizon_summary[model_name][str(h)]['rmse']['mean']
                 for h in HORIZONS}
    for model_name in per_horizon_summary
}).T[[f'{h}min' for h in HORIZONS]].round(2)

print('=== RMSE (mg/dL), mean only -- for a quick ranking scan ===')
display(rmse_means)

=== RMSE (mg/dL), mean only -- for a quick ranking scan ===


,15min,30min,60min,90min,120min
CGM-only,10.70,17.01,24.40,28.88,31.57
Gated_time,10.58,16.68,24.07,28.40,31.25
Gated_resnet18,11.72,17.91,25.82,30.62,33.41
Gated_vit,13.01,18.85,26.55,31.40,34.10
Gated_clip,12.23,18.10,25.75,30.56,33.33
Gated_dinov2,12.26,18.38,26.44,31.24,33.88


In [7]:
OUTPUT_DIR = PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'analysis' / 'results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for metric_key, table in tables.items():
    out_path = OUTPUT_DIR / f'initial_multimodal_val_{metric_key}_table.csv'
    table.to_csv(out_path)
    print(f'saved {out_path}')

rmse_means_path = OUTPUT_DIR / 'initial_multimodal_val_rmse_means.csv'
rmse_means.to_csv(rmse_means_path)
print(f'saved {rmse_means_path}')

saved /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/analysis/results/initial_multimodal_val_rmse_table.csv
saved /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/analysis/results/initial_multimodal_val_mae_table.csv
saved /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/analysis/results/initial_multimodal_val_grmse_table.csv
saved /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/analysis/results/initial_multimodal_val_zone_a_table.csv
saved /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/analysis/results/initial_multimodal_val_zone_ab_table.csv
saved /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/analysis/results/initial_multimodal_val_rmse_means.csv
